# Phase 3: Comprehensive Dataset Audit & Data Quality Analysis
### Smart Grid Cyber-Physical Intrusion Detection

This notebook audits:
1. Data shapes, numerical/categorical data types
2. Missing values and infinity occurrences ($Z = V/I$ in transmission relays)
3. Duplicate records
4. Class balance and distribution
5. Critical data leakage vectors (`Flow ID`, IP addresses, `Attacker.*` logs)


In [ ]:
# --- Environment Setup ---
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.sans-serif': 'Arial', 'font.size': 11})


## 1. Inspecting Dataset 1: `binaryAllNaturalPlusNormalVsAttacks`
Checking impedance infinity values and class imbalance.


In [ ]:
p1 = os.path.join(PROJECT_ROOT, "binaryAllNaturalPlusNormalVsAttacks", "data1.csv")
df1 = pd.read_csv(p1)
print(f"data1.csv Shape: {df1.shape}")
print(f"Target distribution:\n{df1['marker'].value_counts(normalize=True)}")

# Check for infinity values in impedance columns
inf_counts = np.isinf(df1.select_dtypes(include=[np.number])).sum()
print(f"Columns with infinity values:\n{inf_counts[inf_counts > 0]}")


## 2. Inspecting Dataset 4: `CICFlowMeter_Balanced_DNP3_SCADA`
Checking direct leakage vectors (`Flow ID`, `Src IP`, `Dst IP`, `Timestamp`).


In [ ]:
train_p = os.path.join(PROJECT_ROOT, "CICFlowMeter_Training_Balanced.csv")
df_scada = pd.read_csv(train_p)
print(f"DNP3 SCADA Train Shape: {df_scada.shape}")
print(f"Classes ({df_scada['Label'].nunique()}):\n{df_scada['Label'].value_counts()}")

leakage_cols = ['Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Timestamp']
print(f"Direct leakage columns present: {[c for c in leakage_cols if c in df_scada.columns]}")


## 3. Inspecting Dataset 6: `data4cyber_dataset`
Multi-modal DER telemetry (PV solar, battery storage, AC meters, Modbus, MQTT).


In [ ]:
s0_p = os.path.join(PROJECT_ROOT, "data4cyber_dataset", "S0_benign_baseline", "dataset.csv")
df_s0 = pd.read_csv(s0_p)
print(f"Scenario S0 Benign Shape: {df_s0.shape}")
print(f"Sample electrical columns: {[c for c in df_s0.columns if 'p_sum3' in c or 'soc' in c.lower()][:6]}")
print(f"Sample cyber/protocol columns: {[c for c in df_s0.columns if 'modbus' in c.lower() or 'price' in c.lower()][:6]}")


## 4. Visualizing Class Distributions Across Datasets


In [ ]:
from IPython.display import Image, display

# Display generated distribution plots
plots = [
    "reports/plots/data4cyber_dataset/family_distribution.png",
    "reports/plots/CICFlowMeter_Balanced_DNP3_SCADA/class_distribution.png",
    "reports/plots/binaryAllNaturalPlusNormalVsAttacks/class_distribution.png"
]
for p in plots:
    full_p = os.path.join(PROJECT_ROOT, p)
    if os.path.exists(full_p):
        print(f"Plot: {p}")
        display(Image(filename=full_p))
